# AvocadoIA · Entrenamiento en Colab (spec 003)

Lanzador **sin lógica propia**: solo monta Drive, clona la rama e invoca `python -m avocadoia_modelo` (research M17).

**Antes de abrir este notebook (en la PC):**
1. `python -m avocadoia_modelo verificar` y `python -m avocadoia_modelo preparar` (la caché queda en `Mi unidad/AvocadoIA/modelo/cache/`).
2. Commit y push de `datos/modelo/verificacion.json`.

**En Colab:** *Entorno de ejecución → Cambiar tipo → GPU (T4 o superior).*
Todo lo que se escribe (modelos, logits, registros) queda en `MyDrive/AvocadoIA/modelo/`; al terminar, en la PC: `python -m avocadoia_modelo importar`.

In [ ]:
!nvidia-smi -L
import sys; print(sys.version)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Clonar la rama
Si el repositorio es privado, guarde un token de GitHub (solo lectura) en **Secretos de Colab** con el nombre `GITHUB_TOKEN`.

In [ ]:
import os
RAMA = "003-modelo-clasificador"
REPO = "github.com/Teo344/Entregables.git"
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN}@{REPO}" if TOKEN else f"https://{REPO}"
if not os.path.exists("/content/Entregables"):
    !git clone -q -b {RAMA} {URL} /content/Entregables
else:
    !git -C /content/Entregables pull -q
%cd /content/Entregables
!git log --oneline -1

## Instalar dependencias fijadas
Si la celda siguiente informa una versión de TensorFlow distinta de 2.20.0, reinicie el entorno (*Entorno de ejecución → Reiniciar sesión*) y continúe desde la celda de montaje de Drive.

In [ ]:
!pip install -q -e ".[modelo]"
!python -c "import tensorflow as tf, keras; print('TF', tf.__version__, '| Keras', keras.__version__, '| GPU', tf.config.list_physical_devices('GPU'))"

In [ ]:
# Comprobar rutas de Drive y llevar la verificación del repo a la carpeta de datos de Colab.
import os, shutil, json
DRIVE = "/content/drive/MyDrive/AvocadoIA/modelo"
DATOS = f"{DRIVE}/datos_modelo"
V4 = "/content/drive/Othercomputers/Mi portátil/Modelo/Evidencia_Tesis_Aguacate/05_datos/Dataset_split_experimental_v4_20261104_230751"
print("v4 visible:", os.path.isdir(V4))
os.makedirs(DATOS, exist_ok=True)
shutil.copy2("datos/modelo/verificacion.json", DATOS)
ver = json.load(open(f"{DATOS}/verificacion.json", encoding="utf-8"))
print("verificación:", ver["estado"], "| imágenes:", ver["total"], "| conteos:", ver["conteos"])
print("cachés:", os.listdir(f"{DRIVE}/cache") if os.path.isdir(f"{DRIVE}/cache") else "ninguna")

In [ ]:
# Solo si no hay caché (lento: lee las imágenes originales desde Drive).
# !python -m avocadoia_modelo preparar

## Spike de la firma `explain` (US6, T032)
¿Grad-CAM++ funciona dentro del modelo TFLite del teléfono? Usa el modelo ya entrenado (no reentrena).

In [ ]:
# Id de la corrida a analizar: cópielo de datos_modelo/bitacora.csv tras entrenar
CORRIDA = "<id de la corrida>"
# !python -m avocadoia_modelo spike-explain --corrida {CORRIDA}   # ya resuelto con el v4 (M14.1)


## Corrida base con el dataset 002 (T070)
El dataset por defecto es la partición de la especificación 002 (research M20): 2 188 imágenes de `Modelo/PROCESADO`. La caché ya se prepara en la PC (`preparar`). Si Colab se desconecta, vuelva a ejecutar la misma celda: la corrida se reanuda con el mismo id.

In [ ]:
!python -m avocadoia_modelo entrenar --arquitectura mobilenetv3 --semilla 42

### Reproducibilidad (SC-006)
Repetir la misma corrida crea un id nuevo; la diferencia de macro-F1 de prueba debe ser ≤ 0,01 (ver `bitacora.csv`).

In [ ]:
!python -m avocadoia_modelo entrenar --arquitectura mobilenetv3 --semilla 42
import csv
for f in csv.DictReader(open(f"{DATOS}/bitacora.csv", encoding="utf-8")):
    print(f["id"], f["val_macro_f1"], f["test_macro_f1"], f["experimental"])

## Siguientes pasos (se habilitan con las tareas correspondientes)

In [ ]:
# US4 (T045): mapas Grad-CAM++ de aciertos y errores del conjunto de prueba
!python -m avocadoia_modelo explicar --corrida {CORRIDA}

# Próximas (se habilitan con sus tareas):
# US2 (T035):  !python -m avocadoia_modelo comparar --ejecutar
# US3 (T040):  !python -m avocadoia_modelo calibrar --corrida {CORRIDA}
# US5 (T048):  !python -m avocadoia_modelo exportar --corrida {CORRIDA}
